# Assemble MSD serology data for the Sound Life cohort

We have two types of serology data derived from MSD (Meso Scale Diagnostics) assays:

**IgG Serology**: Measurement of the total concentration of IgG antibodies in plasma that are able to bind to specific flu HA antigens. A standard curve of calibration samples are used to convert signal values to concentrations.

**Hemagglutination Inhibition (HAI Assay)**: Measurement of the percent inhibition of HA binding to labeled RBC vessicles. Serum/Plasma-free blank samples are used as a reference, and the fraction of this reference signal that is observed for each plasma treatment sample are used to compute the percent of inhibition. A 1:4 dilution series of reference standard samples are also included, but are not used as a direct standard curve.

In [1]:
from datetime import date

import hisepy
import os
import pandas as pd
import polars as pl
import re

In [224]:
if not os.path.isdir('output'):
    os.mkdir('output')

### Helper functions

In [2]:
def specimens_to_kits(specimens):
    sample_kits = []
    for specimen in specimens:
        if 'PL' in specimen:
            sample_kit = re.sub('PL([0-9]+)-.+','KT\\1', specimen)
            sample_kits.append(sample_kit)
        else:
            sample_kits.append(None)

    return sample_kits

### MSD Assay Metadata

In [3]:
assay_meta_uuid = '5e3a7f65-d92a-4af0-8983-1d31a9000e14'
assay_meta_csv = hisepy.cache_files([assay_meta_uuid])[0]
assay_meta = pl.read_csv(assay_meta_csv)

In [4]:
assay_meta.head()

msd.assayName,msd.assayType,msd.antigenProtein,msd.antigenName,msd.antigenFullName,msd.antigenVirus,msd.antigenStrain,msd.antigenSubtype,msd.antigenIsolate
str,str,str,str,str,str,str,str,str
"""Flu A/Brisbane (H1N1)""","""IgG Serology""","""HA""","""A/Brisbane""","""A/Brisbane/02/2018 (H1N1) pdm0…","""Influenza""","""A""","""H1N1""","""Brisbane"""
"""Flu A/Hong Kong (H3N2)""","""IgG Serology""","""HA""","""A/Hong Kong""","""A/Hong Kong/2671/2019 (H3N2)-l…","""Influenza""","""A""","""H3N2""","""Hong Kong"""
"""Flu A/Michigan (H1N1)""","""IgG Serology""","""HA""","""A/Michigan""","""A/Michigan/45/2015 (H1N1)-like…","""Influenza""","""A""","""H1N1""","""Michigan"""
"""Flu A/Victoria (H1N1)""","""IgG Serology""","""HA""","""A/Victoria""","""A/Victoria/2570/2019 (H1N1) pd…","""Influenza""","""A""","""H1N1""","""Victoria"""
"""Flu B/Colorado HA""","""IgG Serology""","""HA""","""B/Colorado""","""B/Colorado/06/2017-like virus …","""Influenza""","""B""","""Victoria""","""Colorado"""


In [5]:
assay_meta = assay_meta.select(
    ['msd.assayName', 'msd.antigenName', 'msd.antigenSubtype', 'msd.antigenFullName']
)

In [6]:
assay_meta.head()

msd.assayName,msd.antigenName,msd.antigenSubtype,msd.antigenFullName
str,str,str,str
"""Flu A/Brisbane (H1N1)""","""A/Brisbane""","""H1N1""","""A/Brisbane/02/2018 (H1N1) pdm0…"
"""Flu A/Hong Kong (H3N2)""","""A/Hong Kong""","""H3N2""","""A/Hong Kong/2671/2019 (H3N2)-l…"
"""Flu A/Michigan (H1N1)""","""A/Michigan""","""H1N1""","""A/Michigan/45/2015 (H1N1)-like…"
"""Flu A/Victoria (H1N1)""","""A/Victoria""","""H1N1""","""A/Victoria/2570/2019 (H1N1) pd…"
"""Flu B/Colorado HA""","""B/Colorado""","""Victoria""","""B/Colorado/06/2017-like virus …"


### Sample Metadata

In [7]:
meta_uuid = 'af25e3e7-25c1-4476-afb4-926bd201db8f'

In [8]:
meta_file = hisepy.cache_files([meta_uuid])[0]

In [9]:
meta = pl.read_csv(meta_file)

In [10]:
meta.shape

(868, 19)

Add flu year, then drop the drawDate field for drawYear

In [11]:
meta = meta.with_columns(
    pl.when(pl.col('sample.drawDate') < '2020-07')
    .then(pl.lit('2019-2020'))
    .when(pl.col('sample.drawDate') < '2021-07')
    .then(pl.lit('2020-2021'))
    .otherwise(pl.lit('2021-2022'))
    .alias('vaccine.year')
).with_columns(
    pl.col('sample.drawDate').str.replace('-.+','')
).rename({'sample.drawDate':'sample.drawYear'})

In [12]:
baselines = meta.filter(
    pl.col('sample.visitName').str.contains('Flu Year')
).filter(
    pl.col('sample.visitName').str.contains('Day 0')
)

In [13]:
baselines.shape

(176, 20)

In [14]:
meta.head()

,cohort.cohortGuid,subject.subjectGuid,subject.biologicalSex,subject.cmv,subject.bmi,subject.race,subject.ethnicity,subject.birthYear,subject.ageAtFirstDraw,subject.covidVaxDose1.daysSinceFirstVisit,subject.covidVaxDose2.daysSinceFirstVisit,sample.sampleKitGuid,sample.visitName,sample.drawYear,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit,specimen.specimenGuid,pipeline.fileGuid,vaccine.year
i64,str,str,str,str,f64,str,str,i64,i64,f64,f64,str,str,str,i64,i64,str,str,str
0,"""BR1""","""BR1001""","""Female""","""Negative""",23.0,"""Caucasian""","""Non-Hispanic origin""",1987,32,null,null,"""KT00001""","""Flu Year 1 Day 0""","""2019""",32,0,"""PB00001-01""","""fec489f9-9a74-4635-aa91-d2bf09…","""2019-2020"""
1,"""BR1""","""BR1002""","""Male""","""Negative""",22.0,"""Caucasian""","""Non-Hispanic origin""",1991,28,440.0,461.0,"""KT00002""","""Flu Year 1 Day 0""","""2019""",28,0,"""PB00002-01""","""7c0c7979-eebd-4aba-b5b2-6e76b4…","""2019-2020"""
2,"""BR1""","""BR1003""","""Female""","""Negative""",21.0,"""Caucasian""","""Non-Hispanic origin""",1989,30,440.0,461.0,"""KT00003""","""Flu Year 1 Day 0""","""2019""",30,0,"""PB00003-01""","""40efd03a-cb2f-4677-af42-a056cb…","""2019-2020"""
3,"""BR1""","""BR1004""","""Male""","""Negative""",22.0,"""Caucasian""","""Non-Hispanic origin""",1989,30,543.0,563.0,"""KT00004""","""Flu Year 1 Day 0""","""2019""",30,0,"""PB00004-01""","""68fbcd34-1d63-461d-8195-df5b8d…","""2019-2020"""
4,"""BR1""","""BR1005""","""Female""","""Negative""",20.0,"""Caucasian""","""Non-Hispanic origin""",1992,27,451.0,492.0,"""KT00006""","""Flu Year 1 Day 0""","""2019""",27,0,"""PB00006-01""","""ea8d98e9-e99e-4dc6-9e78-9866e0…","""2019-2020"""


In [15]:
age_groups = {
    'BR1': 'Young Adult',
    'BR2': 'Older Adult'
}

In [16]:
meta = meta.with_columns(
    pl.Series(
        name = 'subject.ageGroup',
        values = [age_groups[c] for c in meta['cohort.cohortGuid']]
    )
)

In [17]:
meta.columns

['',
 'cohort.cohortGuid',
 'subject.subjectGuid',
 'subject.biologicalSex',
 'subject.cmv',
 'subject.bmi',
 'subject.race',
 'subject.ethnicity',
 'subject.birthYear',
 'subject.ageAtFirstDraw',
 'subject.covidVaxDose1.daysSinceFirstVisit',
 'subject.covidVaxDose2.daysSinceFirstVisit',
 'sample.sampleKitGuid',
 'sample.visitName',
 'sample.drawYear',
 'sample.subjectAgeAtDraw',
 'sample.daysSinceFirstVisit',
 'specimen.specimenGuid',
 'pipeline.fileGuid',
 'vaccine.year',
 'subject.ageGroup']

In [18]:
keep_meta_cols = [
    'cohort.cohortGuid',
    'subject.subjectGuid',
    'sample.sampleKitGuid',
    'subject.biologicalSex',
    'subject.birthYear',
    'subject.ageAtFirstDraw',
    'subject.ageGroup',
    'subject.race',
    'subject.ethnicity',
    'subject.cmv',
    'sample.visitName',
    'sample.drawYear',
    'sample.subjectAgeAtDraw',
    'sample.daysSinceFirstVisit',
    'vaccine.year'
]

In [19]:
meta = meta.select(keep_meta_cols)

In [20]:
meta.head()

cohort.cohortGuid,subject.subjectGuid,sample.sampleKitGuid,subject.biologicalSex,subject.birthYear,subject.ageAtFirstDraw,subject.ageGroup,subject.race,subject.ethnicity,subject.cmv,sample.visitName,sample.drawYear,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit,vaccine.year
str,str,str,str,i64,i64,str,str,str,str,str,str,i64,i64,str
"""BR1""","""BR1001""","""KT00001""","""Female""",1987,32,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",32,0,"""2019-2020"""
"""BR1""","""BR1002""","""KT00002""","""Male""",1991,28,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",28,0,"""2019-2020"""
"""BR1""","""BR1003""","""KT00003""","""Female""",1989,30,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",30,0,"""2019-2020"""
"""BR1""","""BR1004""","""KT00004""","""Male""",1989,30,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",30,0,"""2019-2020"""
"""BR1""","""BR1005""","""KT00006""","""Female""",1992,27,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",27,0,"""2019-2020"""


### Serology standard ids

In [ ]:
std_uuid = '878ed657-7c57-4b6b-8d2b-da68c243da88'
std_csv = hisepy.cache_files([std_uuid])[0]
std_ids = pl.read_csv(std_csv)

In [141]:
std_ids = pl.read_csv('MSD_standards_and_control_ids_2025-07-30.csv')

In [150]:
std_ids = std_ids.rename({'specimen.specimenGuid': 'Sample'})

## IgG Serology Data

In [172]:
msd_uuid = 'df68e382-a14e-4d86-b5c8-25963c084a54'

In [173]:
all_serology = pl.read_csv(hisepy.cache_files([msd_uuid])[0], infer_schema_length = 10000)
all_serology.shape

(46383, 22)

Convert specimen IDs to sampleKitGuid for matching to sample metadata

In [174]:
all_serology = all_serology.with_columns(
    pl.Series(name = 'sample.sampleKitGuid',
              values = specimens_to_kits(all_serology['Sample']))
)

Clean up batch IDs

In [175]:
all_serology = all_serology.with_columns(
    pl.col('Batch ID').str.replace(',.+', '')
)

Filter for custom panel runs

In [176]:
sample_serology = all_serology.filter(
    pl.col('Notes').str.contains('Custom PLAN-00072')
)
sample_serology.shape

(27720, 23)

Select samples in sample metadata set

In [177]:
sample_serology = sample_serology.filter(
    pl.col('sample.sampleKitGuid').is_in(meta['sample.sampleKitGuid'].unique().to_list())
)
sample_serology.shape

(13272, 23)

Select controls and standards from batches that match our samples

In [178]:
control_serology = all_serology.filter(
    # Keep selected batches
    pl.col('Batch ID').is_in(sample_serology['Batch ID'].unique().to_list())
).filter(
    # Drop samples
    pl.col('sample.sampleKitGuid').is_null()
)
control_serology.shape

(8372, 23)

Add sample names as sample.sampleKitGuid for controls and QC standards

In [179]:
control_serology = control_serology.drop(
    'sample.sampleKitGuid'
).join(
    std_ids,
    how = 'left',
    on = 'Sample'
).select(sample_serology.columns)

Combine samples and controls

In [180]:
serology = pl.concat([sample_serology, control_serology])
serology.shape

(21644, 23)

Is there any difference between Signal and Adjusted Signal?

In [181]:
serology = serology.with_columns(
    (abs(pl.col('Adjusted Signal') - pl.col('Signal'))).alias('diff')
)

In [182]:
max(serology['diff'])

0.0

No difference, so let's use Signal as the primary raw value.

In [183]:
keep_cols = {
    'sample.sampleKitGuid': 'sample.sampleKitGuid',
    'Sample': 'specimen.specimenGuid', 
    'Batch ID': 'msd.batchID',
    'well': 'msd.wellID', 
    'Assay': 'msd.assayName',
    'Dilution': 'msd.sampleDilution', 
    'Concentration': 'msd.standardConc',
    'Signal': 'msd.signalWell', 
    'Mean': 'msd.signalMean', 
    'CV': 'msd.signalCV',
    'Calc. Concentration': 'msd.concWell', 
    'Calc. Conc. Mean': 'msd.concMean', 
    'Calc. Conc. CV': 'msd.concCV'
}

In [184]:
serology = serology.select(keep_cols.keys()).rename(keep_cols)
serology.shape

(21644, 13)

In [191]:
serology = serology.with_columns(
    pl.col('msd.standardConc').cast(pl.Float64).round(6),
    pl.col('msd.concWell').cast(pl.Float64),
    pl.col('msd.concMean').cast(pl.Float64)
)

In [193]:
serology.head()

sample.sampleKitGuid,specimen.specimenGuid,msd.batchID,msd.wellID,msd.assayName,msd.sampleDilution,msd.standardConc,msd.signalWell,msd.signalMean,msd.signalCV,msd.concWell,msd.concMean,msd.concCV
str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""KT00001""","""PL00001-03""","""MSD-00029""","""C09""","""Flu A/Victoria (H1N1)""",10000.0,null,4422.0,4713.0,10.632476,1787.528284,1907.391212,10.809815
"""KT00001""","""PL00001-03""","""MSD-00029""","""C08""","""Flu A/Victoria (H1N1)""",10000.0,null,5292.0,4713.0,10.632476,2145.471789,1907.391212,10.809815
"""KT00001""","""PL00001-03""","""MSD-00029""","""C07""","""Flu A/Victoria (H1N1)""",10000.0,null,4426.0,4713.0,10.632476,1789.173564,1907.391212,10.809815
"""KT00001""","""PL00001-03""","""MSD-00029""","""C09""","""Flu A/Hong Kong (H3N2)""",10000.0,null,12026.0,12162.0,1.038986,6827.241774,6910.345676,1.1202
"""KT00001""","""PL00001-03""","""MSD-00029""","""C08""","""Flu A/Hong Kong (H3N2)""",10000.0,null,12183.0,12162.0,1.038986,6923.392038,6910.345676,1.1202


### Join assay and sample metadata

In [219]:
serology = serology.join(
    meta,
    how = 'left',
    on = 'sample.sampleKitGuid'
).join(
    assay_meta,
    how = 'left',
    on = 'msd.assayName'
)

In [220]:
serology.shape

(21644, 30)

## HAI Assays

#### Calibration/control positions
Calibration samples were placed in the first two columns (in duplicate) in a decreasing 1:4 dilution series from the top row (A) to the bottom (F). The last two rows of these columns (G, H) are blanks, which serve as the positive control for HA binding.

In [218]:
control_rows = ['A','B','C','D','E','F','G','H']
#control_dils = [1,1/4,1/16,1/64,1/256,1/1024,0,0]
control_dils = [1,4,16,64,256,1024,0,0]
control_cols = ['01','02']

control_samples = {}
control_specimens = {}
control_dilutions = {}
for i in range(len(control_rows)):
    row = control_rows[i]
    dil = control_dils[i]
    for col in control_cols:
        rc = row + col
        control_dilutions[rc] = dil
        
        if row in ['G','H']:
            control_samples[rc] = 'Diluent Only Control (Blank)'
            control_specimens[rc] = 'Diluent Only Control (Blank)'
        else:
            control_samples[rc] = 'HAI Reference Standard'
            if row == 'A':
                control_specimens[rc] = 'HAI Reference Standard 1X'
            else:
                control_specimens[rc] = 'HAI Reference Standard 1:' + str(dil)

### Initial HAI experiments
Our first few batches of HAI have a slightly different format from other HAI results. We'll read and structure these for release.

In [33]:
hai_data_uuids = {
    'EXP-01042': '9c100dc9-60a7-4b98-90e7-69277b451c36',
    'EXP-01072': 'd0586978-716c-4df6-9b09-7d8d9f9d5a78',
    'EXP-01111': '3dd84f81-4392-4f66-996a-335975e9b40e'
}

Sample metadata for our pilot experimentn (EXP-01042) - not present in the data table, but can be added based on the well positions.

In [34]:
pilot_sample_info_uuid = '93837e91-cd3a-48cc-9cab-d59f0054b7cd'
pilot_sample_info_xlsx = hisepy.cache_files([pilot_sample_info_uuid])[0]
pilot_sample_info = pl.read_excel(pilot_sample_info_xlsx, sheet_id = 2).head(8)

In [35]:
pilot_sample_info = pilot_sample_info.rename(
    {'__UNNAMED__0': 'row'}
).unpivot(
    index = 'row',
    variable_name = 'col',
    value_name = 'Sample'
).with_columns(
    pl.when(
        pl.col('col').str.len_chars() == 1
    ).then(pl.lit('0')
    ).otherwise(pl.lit('')
    ).alias('pad')
).with_columns(
    pl.concat_str(
        pl.col('row'),
        pl.col('pad'),
        pl.col('col')
    ).alias('Well')
).drop(['row','pad','col'])

In [36]:
pilot_sample_info.head()

Sample,Well
str,str
"""CAL 1""","""A01"""
"""CAL 2""","""B01"""
"""CAL 3""","""C01"""
"""CAL 4""","""D01"""
"""CAL 5""","""E01"""


Read and standardize the HAI experiment batches:

In [208]:
keep_cols = {
    'sample.sampleKitGuid': 'sample.sampleKitGuid',
    'Sample': 'specimen.specimenGuid',
    'Plate Name': 'msd.plateID', 
    'Well': 'msd.wellID', 
    'Assay': 'msd.assayName',
    'Dilution': 'msd.sampleDilution',
    'Signal': 'msd.signalWell',
    'Mean': 'msd.signalMean',
    'CV': 'msd.signalCV'
}

In [215]:
hai_list = []
for batch_id, uuid in hai_data_uuids.items():
    hai_csv = hisepy.cache_files([uuid])[0]
    hai_bn = os.path.basename(hai_csv)
    print(f'Reading {hai_bn}')
    
    # pandas here - for some reason, polars crashes the kernal with these
    all_hai = pd.read_csv(hai_csv, skiprows = 1)
    # but we can convert to polars for other steps
    all_hai = pl.DataFrame(all_hai)
    print(all_hai.shape)
    
    # Convert plate and sample name from the pilot batch
    if batch_id == 'EXP-01042':
        all_hai = all_hai.filter(
            pl.col('Plate Name') == 'Manual_1'
        ).drop('Sample').join(
            pilot_sample_info,
            how = 'left',
            on = 'Well'
        ).with_columns(
            pl.lit('EXP-01042_Plate_1').alias('Plate Name')
        )

    # convert specimens to sample kits
    all_hai = all_hai.with_columns(
        pl.Series(name = 'sample.sampleKitGuid',
                  values = specimens_to_kits(all_hai['Sample']))
    )
    # select and rename columns
    all_hai = all_hai.select(keep_cols.keys()).rename(keep_cols)

        # convert dilutions to integers
    all_hai = all_hai.with_columns(
        pl.col('msd.sampleDilution').cast(pl.Int64())
    )
    
    # filter for samples in metadata
    sample_hai = all_hai.filter(
        pl.col('sample.sampleKitGuid').is_in(meta['sample.sampleKitGuid'].unique().to_list())
    )
    print('Sample data: {s}'.format(s = str(sample_hai.shape)))
    
    # filter for controls and standards
    control_hai = all_hai.filter(
        # Keep selected batches
        pl.col('msd.plateID').is_in(sample_hai['msd.plateID'].unique().to_list()),
        # Select first two columns
        pl.col('msd.wellID').is_in(control_samples.keys())
    ).drop(
        # Drop inconsistent values
        ['sample.sampleKitGuid','specimen.specimenGuid','msd.sampleDilution']
    )
    control_hai = control_hai.with_columns(
        # Incorporate corrected values
        pl.Series(name = 'sample.sampleKitGuid', values = [control_samples[w] for w in control_hai['msd.wellID']]),
        pl.Series(name = 'specimen.specimenGuid', values = [control_specimens[w] for w in control_hai['msd.wellID']]),
        pl.Series(
            name = 'msd.sampleDilution', 
            values = [control_dilutions[w] for w in control_hai['msd.wellID']]
        )
    ).select(sample_hai.columns)
    print('Control data: {s}'.format(s = str(control_hai.shape)))

    # combine data and add batch ID
    hai_data = pl.concat([sample_hai, control_hai])
    hai_data = hai_data.with_columns(
        pl.lit(batch_id).alias('msd.batchID')
    )
    print('Combined data: {s}'.format(s = str(hai_data.shape)))
    hai_list.append(hai_data)

Reading EXP-01111 MSD HAI Data.csv
(2880, 17)
Sample data: (2160, 9)
Control data: (480, 9)
Combined data: (2640, 10)
Reading MSD_HAI_Pilot_DataTable.csv
(3840, 17)
Sample data: (800, 9)
Control data: (160, 9)
Combined data: (960, 10)
Reading EXP-01072 MSD Raw Data_pilot2.csv
(2880, 17)
Sample data: (2400, 9)
Control data: (480, 9)
Combined data: (2880, 10)


In [216]:
hai_list[2].filter(
    pl.col('sample.sampleKitGuid').str.contains('HAI')
).select(['specimen.specimenGuid','sample.sampleKitGuid','msd.sampleDilution']).unique().sort('specimen.specimenGuid')

specimen.specimenGuid,sample.sampleKitGuid,msd.sampleDilution
str,str,i64
"""HAI Reference Standard 1:1024""","""HAI Reference Standard""",1024
"""HAI Reference Standard 1:16""","""HAI Reference Standard""",16
"""HAI Reference Standard 1:256""","""HAI Reference Standard""",256
"""HAI Reference Standard 1:4""","""HAI Reference Standard""",4
"""HAI Reference Standard 1:64""","""HAI Reference Standard""",64
"""HAI Reference Standard 1X""","""HAI Reference Standard""",1


### Additional HAI batches
These were run under a uniform SOP, and have consistent data structure. We'll read and structure these to match data above.

In [225]:
hai_data_uuids_2 = {
    'PLAN-00144-5': 'f855b25a-9562-4134-ad43-b7d2edaff648',
    'PLAN-00144-7': '5f75b742-6385-4308-baf8-5c3c27a4f8a6',
    'PLAN-00144-6': 'f63d6eeb-0e34-4f34-9c21-01b68b562bcb',
    'PLAN-00144-4': 'a3ba8423-c172-4237-a825-a1a646342fd6',
    'PLAN-00144-2': '6b34c7e6-1831-415c-9199-c438105060ce',
    'PLAN-00144-1': 'abea7a06-ac2b-42d7-9002-2ff42a1310b6',
    'PLAN-00144-3': '32bee3f5-5162-4582-9768-2e1d8fdb3c7f'
}

In [226]:
keep_cols = {
    'Sample Kit ID': 'sample.sampleKitGuid',
    'Sample': 'specimen.specimenGuid',
    'Plate.Name': 'msd.plateID', 
    'Well': 'msd.wellID', 
    'Assay': 'msd.assayName',
    #'Dilution': 'msd.sampleDilution', 
    'Signal': 'msd.signalWell', 
    'Mean': 'msd.signalMean', 
    'CV': 'msd.signalCV'
}

In [235]:
hai_list = []
for batch_id, uuid in hai_data_uuids_2.items():
    hai_csv = hisepy.cache_files([uuid])[0]
    hai_bn = os.path.basename(hai_csv)
    print(f'Reading {hai_bn}')

    # pandas here - for some reason, polars crashes the kernal with these
    all_hai = pd.read_csv(hai_csv)
    # but we can convert to polars for other steps
    all_hai = pl.DataFrame(all_hai)
    print(all_hai.shape)

    all_hai = all_hai.select(keep_cols.keys()).rename(keep_cols)

    # filter for samples in metadata
    sample_hai = all_hai.filter(
        pl.col('sample.sampleKitGuid').is_in(meta['sample.sampleKitGuid'].unique().to_list())
    )
    print('Sample data: {s}'.format(s = str(sample_hai.shape)))
    # add sample dilution value
    sample_hai = sample_hai.with_columns(
        pl.Series(name = 'msd.sampleDilution', values = [10000] * sample_hai.shape[0])
    )
    
    # filter for controls and standards
    control_hai = all_hai.filter(
        # Keep selected batches
        pl.col('msd.plateID').is_in(sample_hai['msd.plateID'].unique().to_list()),
        # Select first two columns
        pl.col('msd.wellID').is_in(control_samples.keys())
    ).drop(
        # Drop inconsistent values
        ['sample.sampleKitGuid','specimen.specimenGuid']
    )
    control_hai = control_hai.with_columns(
        # Incorporate corrected values
        pl.Series(name = 'sample.sampleKitGuid', values = [control_samples[w] for w in control_hai['msd.wellID']]),
        pl.Series(name = 'specimen.specimenGuid', values = [control_specimens[w] for w in control_hai['msd.wellID']]),
        pl.Series(
            name = 'msd.sampleDilution', 
            values = [control_dilutions[w] for w in control_hai['msd.wellID']]
        )
    ).select(sample_hai.columns)
    print('Control data: {s}'.format(s = str(control_hai.shape)))
    
    # combine data and add batch ID
    hai_data = pl.concat([sample_hai, control_hai])
    hai_data = hai_data.with_columns(
        pl.col('msd.plateID').str.replace('_Plate.+','').alias('msd.batchID')
    )
    print('Combined data: {s}'.format(s = str(hai_data.shape)))
    hai_list.append(hai_data)

Reading Plan-00144_MSD_HAI_Batch5_Datatable for HISE ingest.csv
(2880, 13)
Sample data: (1960, 8)
Control data: (480, 9)
Combined data: (2440, 10)
Reading Plan-00144_MSD_HAI_Batch7_Datatable for HISE ingest.csv
(2880, 13)
Sample data: (1580, 8)
Control data: (480, 9)
Combined data: (2060, 10)
Reading Plan-00144_MSD_HAI_Batch6_Datatable for HISE ingest.csv
(2880, 13)
Sample data: (2280, 8)
Control data: (480, 9)
Combined data: (2760, 10)
Reading Plan-00144_MSD_HAI_Batch4_Datatable_for_HISE_ingest.csv
(2880, 13)
Sample data: (2260, 8)
Control data: (480, 9)
Combined data: (2740, 10)
Reading Plan-00144_MSD_HAI_Batch2_Datatable_for_HISE_ingest.csv
(2880, 13)
Sample data: (2360, 8)
Control data: (480, 9)
Combined data: (2840, 10)
Reading Plan-00144_MSD_HAI_Batch1_Datatable_for_HISE_ingest.csv
(2880, 13)
Sample data: (2200, 8)
Control data: (480, 9)
Combined data: (2680, 10)
Reading Plan-00144_MSD_HAI_Batch3_Datatable_for_HISE_ingest.csv
(2880, 13)
Sample data: (2300, 8)
Control data: (480, 

In [236]:
all_hai = pl.concat(hai_list)

In [237]:
all_hai.shape

(18300, 10)

In [238]:
all_hai.head()

sample.sampleKitGuid,specimen.specimenGuid,msd.plateID,msd.wellID,msd.assayName,msd.signalWell,msd.signalMean,msd.signalCV,msd.sampleDilution,msd.batchID
str,str,str,str,str,i64,i64,f64,i64,str
"""KT00369""","""PL00369-11""","""2BMACAT013_Batch5_Plate15""","""B08""","""A/Brisbane""",8726,8806,1.276815,10000,"""2BMACAT013_Batch5"""
"""KT00369""","""PL00369-11""","""2BMACAT013_Batch5_Plate15""","""B03""","""A/Brisbane""",8885,8806,1.276815,10000,"""2BMACAT013_Batch5"""
"""KT00369""","""PL00369-11""","""2BMACAT013_Batch5_Plate15""","""B08""","""A/Cambodia""",10949,10928,0.278248,10000,"""2BMACAT013_Batch5"""
"""KT00369""","""PL00369-11""","""2BMACAT013_Batch5_Plate15""","""B03""","""A/Cambodia""",10906,10928,0.278248,10000,"""2BMACAT013_Batch5"""
"""KT00369""","""PL00369-11""","""2BMACAT013_Batch5_Plate15""","""B08""","""A/Guangdong""",36088,40136,14.26176,10000,"""2BMACAT013_Batch5"""


In [44]:
ha2 = pl.read_csv(hisepy.cache_files(['f855b25a-9562-4134-ad43-b7d2edaff648'])[0], infer_schema_length = 10000)

In [45]:
ha2.filter(~pl.col('Sample').str.contains('PL'))

Subject ID,Cohort,Visit Name,Sample Kit ID,Plate.Name,Sample.Group,Sample,Assay,Well,Spot,Signal,Mean,CV
str,str,str,str,str,str,str,str,str,i64,i64,i64,f64
null,null,null,null,"""2BMACAT013_Batch5_Plate15""","""Unknowns""","""BLANK""","""A/Brisbane""","""C07""",1,8486,8360,2.13147
null,null,null,null,"""2BMACAT013_Batch5_Plate15""","""Unknowns""","""BLANK""","""A/Brisbane""","""G12""",1,9412,9439,0.397062
null,null,null,null,"""2BMACAT013_Batch5_Plate15""","""Unknowns""","""BLANK""","""A/Brisbane""","""F12""",1,9580,9313,4.062305
null,null,null,null,"""2BMACAT013_Batch5_Plate15""","""Unknowns""","""BLANK""","""A/Brisbane""","""D12""",1,9037,8885,2.419364
null,null,null,null,"""2BMACAT013_Batch5_Plate15""","""Unknowns""","""BLANK""","""A/Brisbane""","""E12""",1,9084,9375,4.389719
…,…,…,…,…,…,…,…,…,…,…,…,…
null,null,null,null,"""2BMACAD009_Batch5_Plate13""","""Standards""","""STD000006""","""B/Phuket""","""F01""",9,709256,664526,9.519346
null,null,null,null,"""2BMACAD009_Batch5_Plate13""","""Standards""","""STD000006""","""B/Washington""","""F01""",10,524366,410083,39.41167
null,null,null,null,"""2BMACAD009_Batch5_Plate13""","""Standards""","""STD000006""","""B/Washington""","""F02""",10,295800,410083,39.41167
